# modeling_v13 — 2차 선별 (Colab): **Conservative + GA**

M1 다이어트 `Conservative` 셋에 **유전 알고리즘(GA)** 2차 선별. GroupKFold(C20) OOF 를 적합도로 진화.
**고정** core10(10)+필수 5센서 champion(항상 포함, floor 보장), **가변** `Conservative` diet−고정만 GA가 선택.

### ▶ Colab 실행 순서
1. 왼쪽 **파일 패널(폴더 아이콘)** 을 열고, 이 폴더의 **4개 파일을 드래그해 업로드**:
   `v13_select_colab.py`, `v13_fdc_pool_wf_oof.csv.gz`, `core10_meta_wf.csv`, `feature_diet_selected.json`
   (또는 이 `colab_GA` 폴더를 통째로 Google Drive 에 올리고 아래 `BASE` 를 그 경로로 지정)
2. 위에서부터 순서대로 셀 실행. **런타임 유형 = CPU** 로 충분(이 LightGBM 경로는 GPU 이득 없음).
3. 끝나면 `select_result_Conservative_GA.json` 이 생성됨 → 로컬로 내려받아 4개 비교에 사용.

> 두 GA 노트북(Conservative/Balanced)은 독립 → **Colab 세션 2개로 병렬** 실행 가능.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# (필요시) 의존성 — Colab 기본 제공. 오류 시 주석 해제
# !pip install -q lightgbm scikit-learn
import os, json, warnings
warnings.filterwarnings("ignore")
import pandas as pd

BASE = "/content/drive/MyDrive/colab_GA"          # 파일들을 /content 에 업로드했으면 "." (기본). Drive 사용 시 그 폴더 경로로 변경.

# 파일 존재 확인
need = ["v13_select_colab.py", "v13_fdc_pool_wf_oof.csv.gz",
        "core10_meta_wf.csv", "feature_diet_selected.json"]
missing = [f for f in need if not os.path.exists(os.path.join(BASE, f))]
assert not missing, f"업로드 누락: {missing} — 파일 패널에 올려주세요"
print("모든 파일 확인:", need)

모든 파일 확인: ['v13_select_colab.py', 'v13_fdc_pool_wf_oof.csv.gz', 'core10_meta_wf.csv', 'feature_diet_selected.json']


In [3]:
import sys; sys.path.insert(0, BASE)
import v13_select_colab as sc

PRESET = "Conservative"
df, y, groups, g = sc.load(PRESET, base=BASE)
ok, have = sc.floor_ok(g["fixed"])
print("df:", df.shape)
print(f"fixed(core10+champion)={len(g['fixed'])}  prunable(diet-fixed)={len(g['prunable'])}")
print("fixed floor:", ok, have)

df: (11939, 667)
fixed(core10+champion)=15  prunable(diet-fixed)=136
fixed floor: True {'C17': 1, 'C11': 1, 'C31': 1, 'C15': 1, 'C16': 1}


## GA 선별

기본값 `pop=16, gens=8, fit_splits=3`. Colab 이 빠르면 `pop`·`gens` 를 키워 탐색을 넓혀도 됨.
적합도 평가마다 프록시 LGBM(200라운드) OOF 3-fold → fit 수가 많다(수백 회). 진행 로그가 세대별로 찍힌다.

In [7]:
import time; t = time.time()
res = sc.ga_select(df, y, groups, g["fixed"], g["prunable"],
                   pop=32, gens=12, cx=0.6, mut=0.08, elite=2,
                   fit_splits=5, seed=42, verbose=True)
selected = res["best_subset"]
print(f"\nGA: prunable {len(g['prunable'])} -> {len(selected)}  best_GKF={res['best_rmse']:.3f}  evals={res['n_eval']}  ({time.time()-t:.0f}s)")

  GA gen  0  best_GKF=69.043  |mask|=72
  GA gen  1  best_GKF=69.043  |mask|=72
  GA gen  2  best_GKF=69.043  |mask|=72
  GA gen  3  best_GKF=69.043  |mask|=72
  GA gen  4  best_GKF=69.043  |mask|=72
  GA gen  5  best_GKF=69.029  |mask|=87
  GA gen  6  best_GKF=69.010  |mask|=81
  GA gen  7  best_GKF=68.865  |mask|=84
  GA gen  8  best_GKF=68.865  |mask|=84
  GA gen  9  best_GKF=68.865  |mask|=84
  GA gen 10  best_GKF=68.865  |mask|=84
  GA gen 11  best_GKF=68.865  |mask|=84

GA: prunable 136 -> 84  best_GKF=68.865  evals=392  (3082s)


## 최종 OOF — 선택셋 vs baseline(2차선별 없음) · M8_PARAMS·705

In [5]:
baseline = sc.final_report(df, y, groups, g["fixed"], g["prunable"], f"{PRESET}+diet(no-2nd)")
best     = sc.final_report(df, y, groups, g["fixed"], selected,       f"{PRESET}+GA")
tbl = pd.DataFrame([baseline, best])[["label","n_total","n_selected","KFold_OOF","GroupKFold_C20","floor_ok"]]
print(tbl.to_string(index=False)); tbl

                    label  n_total  n_selected  KFold_OOF  GroupKFold_C20  floor_ok
Conservative+diet(no-2nd)      151         136     51.817          70.297      True
          Conservative+GA      113          98     50.686          70.768      True


,label,n_total,n_selected,KFold_OOF,GroupKFold_C20,floor_ok
0,Conservative+diet(no-2nd),151,136,51.817,70.297,True
1,Conservative+GA,113,98,50.686,70.768,True


In [8]:
baseline = sc.final_report(df, y, groups, g["fixed"], g["prunable"], f"{PRESET}+diet(no-2nd)")
best     = sc.final_report(df, y, groups, g["fixed"], selected,       f"{PRESET}+GA")
tbl = pd.DataFrame([baseline, best])[["label","n_total","n_selected","KFold_OOF","GroupKFold_C20","floor_ok"]]
print(tbl.to_string(index=False)); tbl

                    label  n_total  n_selected  KFold_OOF  GroupKFold_C20  floor_ok
Conservative+diet(no-2nd)      151         136     51.817          70.297      True
          Conservative+GA       99          84     50.614          70.355      True


,label,n_total,n_selected,KFold_OOF,GroupKFold_C20,floor_ok
0,Conservative+diet(no-2nd),151,136,51.817,70.297,True
1,Conservative+GA,99,84,50.614,70.355,True


## floor 재확인 · 저장

In [6]:
okb, haveb = sc.floor_ok(best["features"]); assert okb, "FLOOR VIOLATION"
print("final floor:", okb, haveb)
out = {"preset": PRESET, "method": "GA",
       "baseline": {k: baseline[k] for k in ["n_total","KFold_OOF","GroupKFold_C20"]},
       "selected_result": {k: best[k] for k in ["n_total","n_selected","KFold_OOF","GroupKFold_C20","floor_ok"]},
       "selected_features": sorted(selected)}
fn = f"select_result_{PRESET}_GA.json"
json.dump(out, open(fn, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("saved:", fn, "-> 파일 패널에서 다운로드")

final floor: True {'C17': 3, 'C11': 11, 'C31': 3, 'C15': 2, 'C16': 3}
saved: select_result_Conservative_GA.json -> 파일 패널에서 다운로드


In [9]:
okb, haveb = sc.floor_ok(best["features"]); assert okb, "FLOOR VIOLATION"
print("final floor:", okb, haveb)
out = {"preset": PRESET, "method": "GA",
       "baseline": {k: baseline[k] for k in ["n_total","KFold_OOF","GroupKFold_C20"]},
       "selected_result": {k: best[k] for k in ["n_total","n_selected","KFold_OOF","GroupKFold_C20","floor_ok"]},
       "selected_features": sorted(selected)}
fn = f"select_result_{PRESET}_GA.json"
json.dump(out, open(fn, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("saved:", fn, "-> 파일 패널에서 다운로드")

final floor: True {'C17': 4, 'C11': 5, 'C31': 3, 'C15': 3, 'C16': 2}
saved: select_result_Conservative_GA.json -> 파일 패널에서 다운로드


> **다음**: 이 `select_result_*.json` 과 RFECV 결과(로컬)를 모아 GroupKFold(C20) 기준 최적 조합 선택.
> 이후 그 셋을 고정하고 파라미터 재튜닝으로 절대 성능 확정.